# Research 63 — Relative inactivity fresh-fold replication

Repeat notebook 62's exact feature and LightGBM recipe on a fixed fresh three-fold stratified partition (CV seed 2025). Compare the same predeclared 80/20 reference blend. The estimator seed remains fixed at 42. No parameter search, weight tuning, or Test access.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    values = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(values).all() and np.all(values >= 0), filename
    row_sums = values.sum(axis=1, keepdims=True)
    assert np.all(row_sums > 0), filename
    return values / row_sums


def make_features(frame):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (numerator, denominator) in ratios.items():
        x[name] = x[numerator] / (x[denominator].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    x["purchase_collection_intensity"] = np.log1p(x["spending_frequency"].clip(lower=0)) * np.log1p(
        x["rare_items_count"].clip(lower=0)
    )
    vip = x["vip_tier"].map(lambda value: "__MISSING__" if pd.isna(value) else str(int(value)))
    region = x["region"].fillna("__MISSING__").astype(str)
    x["vip_region"] = vip + "__" + region
    x["relative_inactivity_ratio"] = x["days_since_last_login"] / (x["account_age_days"].abs() + 1)
    x["missing_count"] = x.isna().sum(axis=1)
    return x.replace([np.inf, -np.inf], np.nan)


p_base = load_probs("segment-oof.csv")
X = make_features(train)
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=2025).split(X, y))
p_candidate = np.zeros((len(y), 4), dtype=float)
started = perf_counter()
for fold, (fit_idx, valid_idx) in enumerate(folds, start=1):
    x_fit, x_valid = X.iloc[fit_idx].copy(), X.iloc[valid_idx].copy()
    for col in cat_cols:
        categories = sorted(x_fit[col].dropna().unique().tolist())
        dtype = pd.CategoricalDtype(categories=categories)
        x_fit[col] = x_fit[col].astype(dtype)
        x_valid[col] = x_valid[col].astype(dtype)
    model = LGBMClassifier(
        objective="multiclass",
        class_weight="balanced",
        n_estimators=440,
        learning_rate=0.063863707925569,
        num_leaves=59,
        max_depth=4,
        min_child_samples=81,
        reg_lambda=0.12942956115511214,
        colsample_bytree=0.7975990992289793,
        random_state=42,
        n_jobs=2,
        verbosity=-1,
    )
    model.fit(x_fit, y[fit_idx], categorical_feature=cat_cols)
    p_candidate[valid_idx] = model.predict_proba(x_valid)
    fold_score = f1_score(y[valid_idx], p_candidate[valid_idx].argmax(axis=1), average="macro")
    print(f"Fold {fold}: candidate macro F1={fold_score:.6f}", flush=True)

p_blend = 0.8 * p_base + 0.2 * p_candidate
candidates = {
    "Current reference": p_base,
    "Balanced-weight VIP×region + relative inactivity": p_candidate,
    "Fixed 80/20 reference + relative inactivity": p_blend,
}
rows = []
base_folds = []
for _, valid_idx in folds:
    base_folds.append(f1_score(y[valid_idx], p_base[valid_idx].argmax(axis=1), average="macro"))
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for fold, (_, valid_idx) in enumerate(folds, start=1):
        score = f1_score(y[valid_idx], pred[valid_idx], average="macro")
        row[f"fold_{fold}_macro_f1"] = score
        row[f"fold_{fold}_delta_vs_reference"] = score - base_folds[fold - 1]
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-relative-inactivity-seed2025-results.csv", index=False)
for suffix, probs in [("balanced", p_candidate), ("80-20-blend", p_blend)]:
    pd.DataFrame(probs, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
        ROOT / f"predictions/segment-relative-inactivity-seed2025-{suffix}-oof.csv", index=False
    )
print(
    f"Elapsed seconds={perf_counter() - started:.1f}. Train OOF only; Test not loaded; no weights searched."
)

Fold 1: candidate macro F1=0.742704


Fold 2: candidate macro F1=0.742736


Fold 3: candidate macro F1=0.744452


,candidate,macro_f1,weighted_f1,fold_1_macro_f1,fold_1_delta_vs_reference,fold_2_macro_f1,fold_2_delta_vs_reference,fold_3_macro_f1,fold_3_delta_vs_reference
0,Current reference,0.750895,0.764683,0.750797,0.000000,0.749412,0.000000,0.752462,0.000000
1,Balanced-weight VIP×region + relative inactivity,0.743301,0.755218,0.742704,-0.008093,0.742736,-0.006676,0.744452,-0.008010
2,Fixed 80/20 reference + relative inactivity,0.751343,0.764866,0.751694,0.000897,0.749678,0.000267,0.752648,0.000186


Elapsed seconds=85.7. Train OOF only; Test not loaded; no weights searched.
